# Trace the Ace — Feature Engineering

This notebook transforms the model-ready master dataset into fully
processed training and validation files.

### Main Tasks

- Create a leakage-safe train-validation split using `session_id`
- Clean text columns
- Build objective-aware model text
- Prepare twenty numerical transcript features
- Create selected logarithmic features
- Recalculate categorical thresholds using training data only
- Impute and scale structured features
- One-hot encode categorical bands
- Create word-level TF-IDF features
- Create character-level TF-IDF features
- Save all processed matrices and preprocessing assets

The model-training notebook will only load these processed files and
train models. It will not perform additional feature processing.

In [1]:
from pathlib import Path
import gc
import json
import joblib

import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


current_dir = Path.cwd().resolve()
search_locations = [current_dir, *current_dir.parents]

project_root = next((path for path in search_locations if (path / "Trace-The-Race-Dataset").is_dir()), None)

if project_root is None:
    raise FileNotFoundError("Trace-The-Race-Dataset folder was not found.")

dataset_root = project_root / "Trace-The-Race-Dataset"
master_file = dataset_root / "outputs" / "03_master_dataset" / "master_train.parquet"

output_dir = dataset_root / "outputs" / "05_feature_engineering"
output_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(master_file)
df.columns = df.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip().str.lower()

df["response_id"] = df["response_id"].astype("string").str.strip()
df["session_id"] = df["session_id"].astype("string").str.strip()
df["correct"] = pd.to_numeric(df["correct"], errors="raise").astype("int8")

print("Master file    :", master_file)
print("Output folder  :", output_dir)
print("Rows           :", f"{len(df):,}")
print("Columns        :", df.shape[1])
print("Unique sessions:", f"{df['session_id'].nunique():,}")

display(df.head())

Master file    : C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\outputs\03_master_dataset\master_train.parquet
Output folder  : C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\outputs\05_feature_engineering
Rows           : 35,072
Columns        : 36
Unique sessions: 22,821


,response_id,session_id,learning_objective,correct,learning_objective_id,source_file,transcript_text,student_text,tutor_text,total_turns,total_words,session_duration_minutes,turns_per_minute,student_turns,tutor_turns,student_turn_ratio,student_word_ratio,avg_student_words_per_turn,avg_tutor_words_per_turn,student_short_turn_ratio,student_long_turn_ratio,student_numeric_turn_ratio,student_question_ratio,tutor_question_ratio,student_response_after_tutor_question_ratio,speaker_switch_rate,longest_tutor_streak_ratio,longest_student_streak_ratio,background_turn_ratio,session_turn_volume_band,session_duration_band,student_turn_share_band,student_response_length_band,student_short_turn_band,tutor_questioning_band,dialogue_switch_band
0,aaaavsh,bcaufvc,Knowing the value of each digit in numbers wit...,1,dqibnvd,bcaufvc.csv,"[BACKGROUND] [unclear]\n[TUTOR] Miss, I can't ...","Can you hear me? Hello?\nYeah, I hear you. Can...","Miss, I can't hear.\nCan you hear me? [unclear...",330,3405,44.7500,7.3743,150,169,0.4545,0.2822,6.4067,13.6391,0.4533,0.2133,0.3333,0.1333,0.5444,0.9022,0.8491,0.0414,0.0267,0.0333,HIGH,MEDIUM,MEDIUM,LOW,MEDIUM,LOW,HIGH
1,aaabhzi,eyutanf,Adding and subtracting tens to a 2-digit number.,1,eukmzxl,eyutanf.csv,[TUTOR] Yay! Hello!\n[STUDENT] Hello.\n[TUTOR]...,Hello.\nHello.\nI was one minute early.\nI'm o...,Yay! Hello!\nHello.\nHello. [Speaker:Backgroun...,280,5280,44.8500,6.2430,123,144,0.4393,0.3680,15.7967,22.1181,0.3740,0.4228,0.3252,0.1301,0.5556,0.7875,0.8421,0.0208,0.0244,0.0464,MEDIUM,HIGH,MEDIUM,HIGH,LOW,LOW,HIGH
2,aaahpnz,juptkxd,Comparing and ordering fractions by finding a ...,0,fjbqcsv,juptkxd.csv,[BACKGROUND] [unclear]\n[TUTOR] Is it me you a...,Hello.\nHello.\nYes.\nGood. How are you?\nNorm...,"Is it me you are looking for?\nOkay, so did yo...",291,4638,44.9500,6.4739,139,139,0.4777,0.3952,13.1871,18.3669,0.4245,0.3597,0.5108,0.1223,0.5612,0.8846,0.8520,0.0216,0.0360,0.0447,MEDIUM,HIGH,HIGH,HIGH,LOW,LOW,HIGH
3,aaajpom,ntwkcfj,Comparing fractions using reasoning.,0,acvbcev,ntwkcfj.csv,[BACKGROUND] [unclear]\n[TUTOR] Hello?\n[STUDE...,"Hello, Tobias. Can you hear me?\nOkay, that's ...","Hello?\nYes, I can hear you.\nGood. Why are yo...",265,4792,45.0667,5.8802,124,132,0.4679,0.2619,10.1210,25.9167,0.4194,0.2984,0.3790,0.1694,0.7273,0.8646,0.8196,0.0227,0.0323,0.0340,MEDIUM,HIGH,HIGH,HIGH,LOW,HIGH,MEDIUM
4,aaamwux,jqriibm,Counting in multiples.,0,krfuudx,jqriibm.csv,[BACKGROUND] [unclear]\n[TUTOR] Hello.\n[STUDE...,"Hello.\nGood. Okay, how was this week for you?...",Hello.\nHello.\nAre you feeling tired today?\n...,270,2640,35.0333,7.7069,102,153,0.3778,0.2197,5.6863,13.0131,0.5196,0.1667,0.3725,0.1176,0.6797,0.7404,0.7480,0.0392,0.0196,0.0556,MEDIUM,LOW,LOW,LOW,MEDIUM,MEDIUM,LOW


## 1. Define Feature Groups

The master dataset contains identifiers, text fields, numerical transcript
features, and threshold-based categorical features.

The original threshold columns are not used directly because they were
calculated before the train-validation split.

Six categorical bands will be recalculated using training data only.
`session_turn_volume_band` is excluded because the EDA found it weak and
non-monotonic.

In [2]:
id_columns = ["response_id", "session_id"]
target_column = "correct"

numerical_features = [
    "total_turns",
    "total_words",
    "session_duration_minutes",
    "turns_per_minute",
    "student_turns",
    "tutor_turns",
    "student_turn_ratio",
    "student_word_ratio",
    "avg_student_words_per_turn",
    "avg_tutor_words_per_turn",
    "student_short_turn_ratio",
    "student_long_turn_ratio",
    "student_numeric_turn_ratio",
    "student_question_ratio",
    "tutor_question_ratio",
    "student_response_after_tutor_question_ratio",
    "speaker_switch_rate",
    "longest_tutor_streak_ratio",
    "longest_student_streak_ratio",
    "background_turn_ratio",
]

text_columns = ["learning_objective", "transcript_text", "student_text", "tutor_text"]

band_sources = {
    "session_duration_band": "session_duration_minutes",
    "student_turn_share_band": "student_turn_ratio",
    "student_response_length_band": "avg_student_words_per_turn",
    "student_short_turn_band": "student_short_turn_ratio",
    "tutor_questioning_band": "tutor_question_ratio",
    "dialogue_switch_band": "speaker_switch_rate",
}

band_features = list(band_sources.keys())

log_source_features = [
    "avg_student_words_per_turn",
    "avg_tutor_words_per_turn",
    "longest_tutor_streak_ratio",
    "longest_student_streak_ratio",
    "background_turn_ratio",
]

log_features = [f"log_{column}" for column in log_source_features]

for column in numerical_features:
    df[column] = pd.to_numeric(df[column], errors="coerce").replace([np.inf, -np.inf], np.nan)

for column in text_columns:
    df[column] = df[column].fillna("").astype("string").str.replace(r"\s+", " ", regex=True).str.strip()

print("ID columns         :", id_columns)
print("Target column      :", target_column)
print("Numerical features :", len(numerical_features))
print("New band features  :", len(band_features))
print("Log features       :", len(log_features))

ID columns         : ['response_id', 'session_id']
Target column      : correct
Numerical features : 20
New band features  : 6
Log features       : 5


## 2. Group-Based Train-Validation Split

The split is performed using `session_id`.

All responses belonging to one tutoring session remain in the same
partition. This prevents the same transcript from appearing in both
training and validation data.

### Split

- 80% of sessions: training
- 20% of sessions: validation
- Random state: 42

In [3]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)

train_index, validation_index = next(splitter.split(df, y=df[target_column], groups=df["session_id"]))

train_df = df.iloc[train_index].copy().reset_index(drop=True)
validation_df = df.iloc[validation_index].copy().reset_index(drop=True)

train_sessions = set(train_df["session_id"])
validation_sessions = set(validation_df["session_id"])
session_overlap = train_sessions.intersection(validation_sessions)

split_summary = pd.DataFrame({
    "partition": ["Train", "Validation"],
    "rows": [len(train_df), len(validation_df)],
    "unique_sessions": [train_df["session_id"].nunique(), validation_df["session_id"].nunique()],
    "correct_rate": [train_df["correct"].mean(), validation_df["correct"].mean()],
})

print("Session overlap:", len(session_overlap))
display(split_summary)

train_df[["session_id"]].drop_duplicates().to_csv(output_dir / "train_session_ids.csv", index=False)
validation_df[["session_id"]].drop_duplicates().to_csv(output_dir / "validation_session_ids.csv", index=False)
split_summary.to_csv(output_dir / "split_summary.csv", index=False)

Session overlap: 0


,partition,rows,unique_sessions,correct_rate
0,Train,28125,18256,0.6995
1,Validation,6947,4565,0.7146


## 3. Create Model Text

Three objective-aware text fields are created.

### Main Text

```text
[OBJECTIVE] learning objective
[TRANSCRIPT] full tagged transcript

In [4]:
def create_text_features(data):
    data = data.copy()

    for column in text_columns:
        data[column] = data[column].fillna("").astype("string").str.replace(r"\s+", " ", regex=True).str.strip()

    data["model_text"] = "[OBJECTIVE] " + data["learning_objective"] + "\n[TRANSCRIPT] " + data["transcript_text"]
    data["student_model_text"] = "[OBJECTIVE] " + data["learning_objective"] + "\n[STUDENT] " + data["student_text"]
    data["tutor_model_text"] = "[OBJECTIVE] " + data["learning_objective"] + "\n[TUTOR] " + data["tutor_text"]

    return data


train_df = create_text_features(train_df)
validation_df = create_text_features(validation_df)

print("Empty training model text   :", train_df["model_text"].str.strip().eq("").sum())
print("Empty validation model text :", validation_df["model_text"].str.strip().eq("").sum())
print("Average train text words    :", train_df["model_text"].str.split().str.len().mean())

display(train_df[["response_id", "session_id", "model_text"]].head())

Empty training model text   : 0
Empty validation model text : 0
Average train text words    : 3864.1712


,response_id,session_id,model_text
0,aaaavsh,bcaufvc,[OBJECTIVE] Knowing the value of each digit in...
1,aaahpnz,juptkxd,[OBJECTIVE] Comparing and ordering fractions b...
2,aaajpom,ntwkcfj,[OBJECTIVE] Comparing fractions using reasonin...
3,aaamwux,jqriibm,[OBJECTIVE] Counting in multiples.\n[TRANSCRIP...
4,aaaqjzc,iykbtex,[OBJECTIVE] Seeing fractions as numbers.\n[TRA...


## 4. Numerical Feature Engineering

The twenty original numerical features are retained.

Two structured feature versions will be created:

### Basic Structured Features

- Twenty original numerical features
- Six categorical bands

### Engineered Structured Features

- Training-only 1st and 99th percentile clipping
- Twenty clipped numerical features
- Five logarithmic features
- Six categorical bands

The clipping limits are fitted only on the training partition and then
applied unchanged to validation data.

In [5]:
clip_limits = {}

train_clipped = pd.DataFrame(index=train_df.index)
validation_clipped = pd.DataFrame(index=validation_df.index)

for column in numerical_features:
    train_values = pd.to_numeric(train_df[column], errors="coerce")
    validation_values = pd.to_numeric(validation_df[column], errors="coerce")

    lower_limit = float(train_values.quantile(0.01))
    upper_limit = float(train_values.quantile(0.99))

    clip_limits[column] = {"lower": lower_limit, "upper": upper_limit}

    train_clipped[column] = train_values.clip(lower=lower_limit, upper=upper_limit)
    validation_clipped[column] = validation_values.clip(lower=lower_limit, upper=upper_limit)

for source_column in log_source_features:
    log_column = f"log_{source_column}"

    train_df[log_column] = np.log1p(train_clipped[source_column].clip(lower=0))
    validation_df[log_column] = np.log1p(validation_clipped[source_column].clip(lower=0))

engineered_numerical_features = numerical_features + log_features

train_engineered_numeric = pd.concat([train_clipped, train_df[log_features]], axis=1)
validation_engineered_numeric = pd.concat([validation_clipped, validation_df[log_features]], axis=1)

with open(output_dir / "clip_limits.json", "w", encoding="utf-8") as file:
    json.dump(clip_limits, file, indent=2)

print("Original numerical features  :", len(numerical_features))
print("Engineered numerical features:", len(engineered_numerical_features))

display(train_df[log_features].head())

Original numerical features  : 20
Engineered numerical features: 25


,log_avg_student_words_per_turn,log_avg_tutor_words_per_turn,log_longest_tutor_streak_ratio,log_longest_student_streak_ratio,log_background_turn_ratio
0,2.0024,2.6837,0.0406,0.0263,0.0328
1,2.6523,2.9636,0.0214,0.0353,0.0437
2,2.4088,3.2927,0.0225,0.0317,0.0334
3,1.9001,2.6400,0.0385,0.0194,0.0541
4,2.4111,3.1637,0.0616,0.0408,0.0430


## 5. Recalculate Threshold Bands

The categorical thresholds are fitted using training rows only.

For every source feature:

- Values at or below the training 33rd percentile become `LOW`
- Values above the 33rd percentile and at or below the 67th percentile
  become `MEDIUM`
- Values above the 67th percentile become `HIGH`

The same saved thresholds are applied to validation data.

In [8]:
def apply_band(series, low_threshold, high_threshold, fill_value):
    values = pd.to_numeric(series, errors="coerce").fillna(fill_value)
    labels = np.select([values <= low_threshold, values <= high_threshold], ["LOW", "MEDIUM"], default="HIGH")
    return pd.Series(labels, index=series.index, dtype="string")


band_thresholds = {}

for band_column, source_column in band_sources.items():
    training_values = pd.to_numeric(train_df[source_column], errors="coerce")
    fill_value = float(training_values.median())

    q33 = float(training_values.quantile(1 / 3))
    q67 = float(training_values.quantile(2 / 3))

    if q33 >= q67:
        positive_values = training_values[training_values > q33]

        if len(positive_values) > 0:
            q67 = float(positive_values.median())
        else:
            q67 = q33

    band_thresholds[band_column] = {
        "source_feature": source_column,
        "low_upper_threshold": q33,
        "medium_upper_threshold": q67,
        "fill_value": fill_value,
    }

    train_df[band_column] = apply_band(train_df[source_column], q33, q67, fill_value)
    validation_df[band_column] = apply_band(validation_df[source_column], q33, q67, fill_value)

with open(output_dir / "band_thresholds.json", "w", encoding="utf-8") as file:
    json.dump(band_thresholds, file, indent=2)

band_distribution = []

for column in band_features:
    train_counts = train_df[column].value_counts().reindex(["LOW", "MEDIUM", "HIGH"], fill_value=0)

    for band, count in train_counts.items():
        band_distribution.append({"feature": column, "band": band, "train_count": int(count)})

band_distribution = pd.DataFrame(band_distribution)
band_distribution.to_csv(output_dir / "band_distribution.csv", index=False)

display(band_distribution)

,feature,band,train_count
0,session_duration_band,LOW,9396
1,session_duration_band,MEDIUM,9382
2,session_duration_band,HIGH,9347
3,student_turn_share_band,LOW,9375
4,student_turn_share_band,MEDIUM,9382
5,student_turn_share_band,HIGH,9368
6,student_response_length_band,LOW,9376
7,student_response_length_band,MEDIUM,9374
8,student_response_length_band,HIGH,9375
9,student_short_turn_band,LOW,9388


## 6. Create Fully Processed Structured Matrices

All preprocessing is fitted on training data only.

### Basic Matrix

Numerical processing:

- Median imputation
- Standard scaling

Categorical processing:

- One-hot encoding
- Unknown-category handling

### Engineered Matrix

Numerical processing:

- Training-only clipping
- Logarithmic features
- Median imputation
- Robust scaling

The final matrices are saved in sparse `.npz` format.

In [9]:
basic_imputer = SimpleImputer(strategy="median")
basic_scaler = StandardScaler()

X_train_basic_numeric = basic_imputer.fit_transform(train_df[numerical_features])
X_validation_basic_numeric = basic_imputer.transform(validation_df[numerical_features])

X_train_basic_numeric = basic_scaler.fit_transform(X_train_basic_numeric).astype(np.float32)
X_validation_basic_numeric = basic_scaler.transform(X_validation_basic_numeric).astype(np.float32)


try:
    band_encoder = OneHotEncoder(categories=[["LOW", "MEDIUM", "HIGH"]] * len(band_features), handle_unknown="ignore", sparse_output=True, dtype=np.float32)
except TypeError:
    band_encoder = OneHotEncoder(categories=[["LOW", "MEDIUM", "HIGH"]] * len(band_features), handle_unknown="ignore", sparse=True, dtype=np.float32)

X_train_bands = band_encoder.fit_transform(train_df[band_features])
X_validation_bands = band_encoder.transform(validation_df[band_features])

X_train_structured_basic = sparse.hstack([sparse.csr_matrix(X_train_basic_numeric), X_train_bands], format="csr")
X_validation_structured_basic = sparse.hstack([sparse.csr_matrix(X_validation_basic_numeric), X_validation_bands], format="csr")


engineered_imputer = SimpleImputer(strategy="median")
engineered_scaler = RobustScaler()

X_train_engineered_numeric = engineered_imputer.fit_transform(train_engineered_numeric[engineered_numerical_features])
X_validation_engineered_numeric = engineered_imputer.transform(validation_engineered_numeric[engineered_numerical_features])

X_train_engineered_numeric = engineered_scaler.fit_transform(X_train_engineered_numeric).astype(np.float32)
X_validation_engineered_numeric = engineered_scaler.transform(X_validation_engineered_numeric).astype(np.float32)

X_train_structured_engineered = sparse.hstack([sparse.csr_matrix(X_train_engineered_numeric), X_train_bands], format="csr")
X_validation_structured_engineered = sparse.hstack([sparse.csr_matrix(X_validation_engineered_numeric), X_validation_bands], format="csr")


band_feature_names = band_encoder.get_feature_names_out(band_features).tolist()
basic_feature_names = numerical_features + band_feature_names
engineered_feature_names = engineered_numerical_features + band_feature_names

sparse.save_npz(output_dir / "train_structured_basic.npz", X_train_structured_basic, compressed=True)
sparse.save_npz(output_dir / "validation_structured_basic.npz", X_validation_structured_basic, compressed=True)

sparse.save_npz(output_dir / "train_structured_engineered.npz", X_train_structured_engineered, compressed=True)
sparse.save_npz(output_dir / "validation_structured_engineered.npz", X_validation_structured_engineered, compressed=True)

joblib.dump({
    "imputer": basic_imputer,
    "scaler": basic_scaler,
    "band_encoder": band_encoder,
    "numerical_features": numerical_features,
    "band_features": band_features,
}, output_dir / "structured_basic_preprocessor.joblib")

joblib.dump({
    "imputer": engineered_imputer,
    "scaler": engineered_scaler,
    "band_encoder": band_encoder,
    "numerical_features": engineered_numerical_features,
    "band_features": band_features,
    "clip_limits": clip_limits,
    "log_source_features": log_source_features,
}, output_dir / "structured_engineered_preprocessor.joblib")

pd.DataFrame({"feature_name": basic_feature_names}).to_csv(output_dir / "structured_basic_feature_names.csv", index=False)
pd.DataFrame({"feature_name": engineered_feature_names}).to_csv(output_dir / "structured_engineered_feature_names.csv", index=False)

print("Basic train matrix      :", X_train_structured_basic.shape)
print("Basic validation matrix :", X_validation_structured_basic.shape)
print("Engineered train matrix :", X_train_structured_engineered.shape)
print("Engineered valid matrix :", X_validation_structured_engineered.shape)

Basic train matrix      : (28125, 38)
Basic validation matrix : (6947, 38)
Engineered train matrix : (28125, 43)
Engineered valid matrix : (6947, 43)


## 7. Word TF-IDF Features

Word-level TF-IDF is fitted on the training `model_text`.

### Configuration

- Unigrams and bigrams
- Minimum document frequency of 2
- Maximum 60,000 features
- Sublinear term frequency
- Single-character tokens retained for mathematical expressions

The fitted vectorizer and transformed sparse matrices are saved.

In [10]:
word_vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.995,
    max_features=60000,
    sublinear_tf=True,
    dtype=np.float32,
)

X_train_word = word_vectorizer.fit_transform(train_df["model_text"])
X_validation_word = word_vectorizer.transform(validation_df["model_text"])

sparse.save_npz(output_dir / "train_word_tfidf.npz", X_train_word, compressed=True)
sparse.save_npz(output_dir / "validation_word_tfidf.npz", X_validation_word, compressed=True)

joblib.dump(word_vectorizer, output_dir / "word_tfidf_vectorizer.joblib")

word_feature_names = word_vectorizer.get_feature_names_out()
pd.DataFrame({"feature_name": word_feature_names}).to_csv(output_dir / "word_tfidf_feature_names.csv", index=False)

print("Word train matrix      :", X_train_word.shape)
print("Word validation matrix :", X_validation_word.shape)
print("Word vocabulary size   :", len(word_feature_names))

Word train matrix      : (28125, 60000)
Word validation matrix : (6947, 60000)
Word vocabulary size   : 60000


## 8. Character TF-IDF Features

Character TF-IDF captures:

- Spelling variation
- Short mathematical expressions
- Partial words
- Punctuation patterns
- Local phrase structure

### Configuration

- Character n-grams from 3 to 5
- Word-boundary-aware character analysis
- Minimum document frequency of 3
- Maximum 60,000 features

This cell may take longer than the word TF-IDF cell.

In [11]:
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    lowercase=True,
    ngram_range=(3, 5),
    min_df=3,
    max_features=60000,
    sublinear_tf=True,
    dtype=np.float32,
)

X_train_char = char_vectorizer.fit_transform(train_df["model_text"])
X_validation_char = char_vectorizer.transform(validation_df["model_text"])

sparse.save_npz(output_dir / "train_char_tfidf.npz", X_train_char, compressed=True)
sparse.save_npz(output_dir / "validation_char_tfidf.npz", X_validation_char, compressed=True)

joblib.dump(char_vectorizer, output_dir / "char_tfidf_vectorizer.joblib")

char_feature_names = char_vectorizer.get_feature_names_out()
pd.DataFrame({"feature_name": char_feature_names}).to_csv(output_dir / "char_tfidf_feature_names.csv", index=False)

print("Character train matrix      :", X_train_char.shape)
print("Character validation matrix :", X_validation_char.shape)
print("Character vocabulary size   :", len(char_feature_names))

gc.collect()

Character train matrix      : (28125, 60000)
Character validation matrix : (6947, 60000)
Character vocabulary size   : 60000


0

## 9. Save Processed Training and Validation Tables

The processed Parquet files contain:

- Response and session identifiers
- Binary target
- Original text fields
- Objective-aware model text
- Twenty original numerical features
- Five logarithmic features
- Six leakage-safe categorical bands

The sparse model matrices are stored separately to avoid placing large
TF-IDF arrays inside Parquet files.

In [12]:
processed_columns = [
    "response_id",
    "session_id",
    "correct",
    "learning_objective",
    "transcript_text",
    "student_text",
    "tutor_text",
    "model_text",
    "student_model_text",
    "tutor_model_text",
] + numerical_features + log_features + band_features

train_processed = train_df[processed_columns].copy()
validation_processed = validation_df[processed_columns].copy()

train_processed.to_parquet(output_dir / "train_processed.parquet", index=False, engine="pyarrow", compression="snappy")
validation_processed.to_parquet(output_dir / "validation_processed.parquet", index=False, engine="pyarrow", compression="snappy")

train_metadata = train_df[["response_id", "session_id", "correct", "learning_objective"]].copy()
validation_metadata = validation_df[["response_id", "session_id", "correct", "learning_objective"]].copy()

train_metadata.to_parquet(output_dir / "train_metadata.parquet", index=False, engine="pyarrow", compression="snappy")
validation_metadata.to_parquet(output_dir / "validation_metadata.parquet", index=False, engine="pyarrow", compression="snappy")

y_train = train_df["correct"].to_numpy(dtype=np.int8)
y_validation = validation_df["correct"].to_numpy(dtype=np.int8)

np.save(output_dir / "train_labels.npy", y_train)
np.save(output_dir / "validation_labels.npy", y_validation)

feature_config = {
    "random_state": 42,
    "split_method": "GroupShuffleSplit",
    "validation_size": 0.20,
    "group_column": "session_id",
    "target_column": "correct",
    "main_text_column": "model_text",
    "id_columns": id_columns,
    "numerical_features": numerical_features,
    "log_features": log_features,
    "engineered_numerical_features": engineered_numerical_features,
    "band_features": band_features,
    "excluded_band_feature": "session_turn_volume_band",
    "train_rows": len(train_df),
    "validation_rows": len(validation_df),
    "train_sessions": train_df["session_id"].nunique(),
    "validation_sessions": validation_df["session_id"].nunique(),
    "train_correct_rate": float(train_df["correct"].mean()),
    "validation_correct_rate": float(validation_df["correct"].mean()),
    "structured_basic_features": len(basic_feature_names),
    "structured_engineered_features": len(engineered_feature_names),
    "word_tfidf_features": int(X_train_word.shape[1]),
    "char_tfidf_features": int(X_train_char.shape[1]),
}

with open(output_dir / "feature_config.json", "w", encoding="utf-8") as file:
    json.dump(feature_config, file, indent=2)

print("Processed train rows      :", f"{len(train_processed):,}")
print("Processed validation rows :", f"{len(validation_processed):,}")
print("Configuration saved       :", output_dir / "feature_config.json")

Processed train rows      : 28,125
Processed validation rows : 6,947
Configuration saved       : C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\outputs\05_feature_engineering\feature_config.json


## 10. Final Validation and Output Manifest

This final step confirms that:

- Training and validation sessions do not overlap
- Matrix rows match the saved labels
- Processed Parquet rows match the sparse matrices
- All expected artifacts were successfully created

A manifest containing file names and sizes is saved in the output folder.

In [13]:
checks = {
    "session_overlap": len(set(train_df["session_id"]).intersection(set(validation_df["session_id"]))),
    "train_processed_rows": len(train_processed),
    "validation_processed_rows": len(validation_processed),
    "train_label_rows": len(y_train),
    "validation_label_rows": len(y_validation),
    "train_structured_basic_rows": X_train_structured_basic.shape[0],
    "validation_structured_basic_rows": X_validation_structured_basic.shape[0],
    "train_structured_engineered_rows": X_train_structured_engineered.shape[0],
    "validation_structured_engineered_rows": X_validation_structured_engineered.shape[0],
    "train_word_rows": X_train_word.shape[0],
    "validation_word_rows": X_validation_word.shape[0],
    "train_char_rows": X_train_char.shape[0],
    "validation_char_rows": X_validation_char.shape[0],
}

validation_report = pd.DataFrame({"check": checks.keys(), "value": checks.values()})
validation_report.to_csv(output_dir / "feature_engineering_validation.csv", index=False)

expected_train_rows = len(train_df)
expected_validation_rows = len(validation_df)

assert checks["session_overlap"] == 0
assert checks["train_structured_basic_rows"] == expected_train_rows
assert checks["train_structured_engineered_rows"] == expected_train_rows
assert checks["train_word_rows"] == expected_train_rows
assert checks["train_char_rows"] == expected_train_rows
assert checks["validation_structured_basic_rows"] == expected_validation_rows
assert checks["validation_structured_engineered_rows"] == expected_validation_rows
assert checks["validation_word_rows"] == expected_validation_rows
assert checks["validation_char_rows"] == expected_validation_rows

manifest_rows = []

for file_path in sorted(output_dir.iterdir()):
    if file_path.is_file():
        manifest_rows.append({
            "file_name": file_path.name,
            "file_type": file_path.suffix,
            "size_mb": file_path.stat().st_size / 1024**2,
        })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(output_dir / "feature_engineering_manifest.csv", index=False)

print("Feature engineering completed successfully.")
print("Output folder     :", output_dir)
print("Train rows        :", f"{len(train_df):,}")
print("Validation rows   :", f"{len(validation_df):,}")
print("Train target rate :", f"{train_df['correct'].mean():.4f}")
print("Valid target rate :", f"{validation_df['correct'].mean():.4f}")

display(validation_report)
display(manifest.round(3))

Feature engineering completed successfully.
Output folder     : C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\outputs\05_feature_engineering
Train rows        : 28,125
Validation rows   : 6,947
Train target rate : 0.6995
Valid target rate : 0.7146


,check,value
0,session_overlap,0
1,train_processed_rows,28125
2,validation_processed_rows,6947
3,train_label_rows,28125
4,validation_label_rows,6947
5,train_structured_basic_rows,28125
6,validation_structured_basic_rows,6947
7,train_structured_engineered_rows,28125
8,validation_structured_engineered_rows,6947
9,train_word_rows,28125


,file_name,file_type,size_mb
0,band_distribution.csv,.csv,0.0010
1,band_thresholds.json,.json,0.0010
2,char_tfidf_feature_names.csv,.csv,0.3830
3,char_tfidf_vectorizer.joblib,.joblib,1.7420
4,clip_limits.json,.json,0.0020
5,feature_config.json,.json,0.0020
6,feature_engineering_validation.csv,.csv,0.0000
7,split_summary.csv,.csv,0.0000
8,structured_basic_feature_names.csv,.csv,0.0010
9,structured_basic_preprocessor.joblib,.joblib,0.0050
